# 📖 Notebook 1: URL Encoding & Hash Generation

The core challenge of a URL shortener is turning a long URL into a short, unique code.
In this notebook we'll explore **three approaches** — from naive to production-ready — and
understand the tradeoffs of each.

## Learning Objectives

By the end of this notebook you'll understand:
- Why simple truncation fails
- How hash functions (MD5, SHA-256) produce fixed-length output
- What Base62 encoding is and why URLs use it instead of Base64
- How an atomic counter in Redis **guarantees** uniqueness with zero collisions
- The tradeoffs between hashing and counter-based approaches

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/bitly
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `bitly_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [1]:
import psycopg2
import redis
import hashlib
import string
import time

# Database connection settings
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "bitly_demo",
    "user": "demo",
    "password": "demo"
}

# Redis connection settings
REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

# Test both connections
try:
    conn = get_db_connection()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker-compose up -d")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")
    print("   Run: docker-compose up -d")

✅ Connected to PostgreSQL
✅ Connected to Redis


## 🤔 The Problem: Long URLs → Short Codes

A URL like `https://www.example.com/articles/2024/system-design-interviews?ref=homepage&utm_source=newsletter`
needs to become something like `short.ly/abc123`.

We need a function that:
1. Produces a **short** output (6–8 characters)
2. Is **unique** — no two different URLs get the same code
3. Is **fast** — we might generate millions of codes per day

Let's explore three approaches and see why only the last one truly works at scale.

---
## ❌ Approach 1: Naive Truncation

The simplest idea: just take the first N characters of the URL as the short code.

In [2]:
def naive_truncate(url: str, length: int = 8) -> str:
    """Take the first N characters of the URL as the short code."""
    return url[:length]

# Try it with a few URLs
urls = [
    "https://www.linkedin.com/in/person-a/",
    "https://www.linkedin.com/in/person-b/",
    "https://www.linkedin.com/in/person-c/",
    "https://www.google.com/search?q=system+design",
]

print("Naive Truncation (first 8 chars):")
print("=" * 60)
for url in urls:
    code = naive_truncate(url)
    print(f"  {code}  ←  {url}")

print()
print("❌ Problem: The first three URLs all produce the SAME code!")
print("   This approach has no randomness — collisions are guaranteed.")

Naive Truncation (first 8 chars):
  https://  ←  https://www.linkedin.com/in/person-a/
  https://  ←  https://www.linkedin.com/in/person-b/
  https://  ←  https://www.linkedin.com/in/person-c/
  https://  ←  https://www.google.com/search?q=system+design

❌ Problem: The first three URLs all produce the SAME code!
   This approach has no randomness — collisions are guaranteed.


---
## ⚠️ Approach 2: Hash Function + Base62 Encoding

A **hash function** takes any input and produces a fixed-size output that *looks random*.
The same input always gives the same output (deterministic), but different inputs almost
always give different outputs.

**Why Base62?**  
Base62 uses characters `a-z`, `A-Z`, `0-9` — that's 62 characters.  
We skip `+` and `/` (from Base64) because `/` is a URL path separator and `+` can be
interpreted as a space in query strings.

In [3]:
# ── Base62 encoder ──────────────────────────────────────────
# Base62 alphabet: 0-9, a-z, A-Z
BASE62_ALPHABET = string.digits + string.ascii_lowercase + string.ascii_uppercase

def base62_encode(number: int) -> str:
    """
    Convert a positive integer to a Base62 string.
    
    Think of it like converting decimal to hex, but with 62 symbols
    instead of 16. Each 'digit' can be 0-9, a-z, or A-Z.
    """
    if number == 0:
        return BASE62_ALPHABET[0]
    
    result = []
    while number > 0:
        number, remainder = divmod(number, 62)
        result.append(BASE62_ALPHABET[remainder])
    
    return ''.join(reversed(result))

# Quick demo of base62
print("Base62 Encoding Examples:")
print("=" * 40)
for num in [0, 61, 62, 1000, 1_000_000, 1_000_000_000]:
    encoded = base62_encode(num)
    print(f"  {num:>15,}  →  {encoded}")

print()
print("💡 1 billion only needs 6 characters in Base62!")
print(f"   62^6 = {62**6:,} possible codes (≈ 56 billion)")
print(f"   62^7 = {62**7:,} possible codes (≈ 3.5 trillion)")

Base62 Encoding Examples:
                0  →  0
               61  →  Z
               62  →  10
            1,000  →  g8
        1,000,000  →  4c92
    1,000,000,000  →  15FTGg

💡 1 billion only needs 6 characters in Base62!
   62^6 = 56,800,235,584 possible codes (≈ 56 billion)
   62^7 = 3,521,614,606,208 possible codes (≈ 3.5 trillion)


### Base62 Is Reversible — Encode and Decode

Because Base62 is just a different number base, you can always **decode** a short
code back to the integer it represents. That's how the *counter approach* (next
section) works: the short code is a compact representation of a unique counter
value. The database lookup by `short_code` is what maps it to the long URL.

If you've ever converted decimal → hex, this is the same idea with 62 symbols
instead of 16.


In [4]:
def base62_decode(code: str) -> int:
    """Convert a Base62 string back to the integer it represents."""
    number = 0
    for char in code:
        number = number * 62 + BASE62_ALPHABET.index(char)
    return number

# Roundtrip: encode then decode must give back the original number
print("Base62 Roundtrip (encode → decode):")
print("=" * 50)
for num in [0, 1, 61, 62, 12345, 1_000_000, 999_999_999_999]:
    encoded = base62_encode(num)
    decoded = base62_decode(encoded)
    ok = "✅" if decoded == num else "❌"
    print(f"  {ok}  {num:>15,}  →  {encoded:<10}  →  {decoded:,}")


Base62 Roundtrip (encode → decode):
  ✅                0  →  0           →  0
  ✅                1  →  1           →  1
  ✅               61  →  Z           →  61
  ✅               62  →  10          →  62
  ✅           12,345  →  3d7         →  12,345
  ✅        1,000,000  →  4c92        →  1,000,000
  ✅  999,999,999,999  →  hBxM5A3     →  999,999,999,999


In [5]:
def hash_and_encode(url: str, length: int = 7) -> str:
    """
    Hash the URL with SHA-256, then Base62-encode the result.
    Take only the first `length` characters as the short code.
    
    Steps:
    1. SHA-256 produces a 256-bit (32-byte) hash
    2. Convert those bytes to a big integer
    3. Base62-encode the integer
    4. Take the first N characters
    """
    hash_bytes = hashlib.sha256(url.encode()).digest()
    hash_int = int.from_bytes(hash_bytes, 'big')
    encoded = base62_encode(hash_int)
    return encoded[:length]

# Try it with the same URLs that broke truncation
print("Hash + Base62 Encoding (SHA-256, 7 chars):")
print("=" * 60)
for url in urls:
    code = hash_and_encode(url)
    print(f"  {code}  ←  {url}")

print()
print("✅ Different URLs now produce different codes!")
print("⚠️  But collisions are still possible (just rare).")

Hash + Base62 Encoding (SHA-256, 7 chars):
  dzt61zq  ←  https://www.linkedin.com/in/person-a/
  rNxgmmm  ←  https://www.linkedin.com/in/person-b/
  rEzVS3n  ←  https://www.linkedin.com/in/person-c/
  KrpinVa  ←  https://www.google.com/search?q=system+design

✅ Different URLs now produce different codes!
⚠️  But collisions are still possible (just rare).


In [6]:
# Let's test: how many URLs before we get a collision?
import random

def collision_test(code_length: int, max_urls: int = 500_000) -> int:
    """Generate random URLs and count how many until a collision."""
    seen = set()
    for i in range(max_urls):
        fake_url = f"https://example.com/{random.randint(0, 10**15)}"
        code = hash_and_encode(fake_url, length=code_length)
        if code in seen:
            return i  # collision at the i-th URL
        seen.add(code)
    return max_urls  # no collision found

print("Collision Test (hash approach):")
print("=" * 50)
for length in [4, 5, 6, 7]:
    code_space = 62 ** length
    collision_at = collision_test(length)
    print(f"  {length} chars → code space {code_space:>15,} → first collision around URL #{collision_at:,}")

print()
print("💡 Shorter codes = more collisions. This is the Birthday Paradox!")
print("   With N possible codes, expect a collision after ~√N URLs.")
print("   A 7-char code (3.5T possibilities) still collides eventually.")

Collision Test (hash approach):
  4 chars → code space      14,776,336 → first collision around URL #5,415
  5 chars → code space     916,132,832 → first collision around URL #13,438


  6 chars → code space  56,800,235,584 → first collision around URL #497,052


  7 chars → code space 3,521,614,606,208 → first collision around URL #500,000

💡 Shorter codes = more collisions. This is the Birthday Paradox!
   With N possible codes, expect a collision after ~√N URLs.
   A 7-char code (3.5T possibilities) still collides eventually.


### Handling Hash Collisions

When a collision happens (two URLs get the same code), we can:
1. Add a random salt and re-hash
2. Retry a few times (bounded retries: 3–5 attempts)
3. Fall back to a different strategy

The database enforces uniqueness via a `UNIQUE` constraint on `short_code`.

In [7]:
def shorten_with_hash(long_url: str, max_retries: int = 5) -> str:
    """
    Shorten a URL using hash + base62, with collision retry logic.
    
    On collision (UNIQUE violation), add a salt and retry.
    """
    conn = get_db_connection()
    conn.autocommit = True
    cursor = conn.cursor()
    
    for attempt in range(max_retries):
        # On retry, add a salt to change the hash output
        url_to_hash = long_url if attempt == 0 else f"{long_url}:{attempt}"
        short_code = hash_and_encode(url_to_hash)
        
        try:
            cursor.execute(
                "INSERT INTO urls (short_code, long_url) VALUES (%s, %s) RETURNING short_code",
                (short_code, long_url)
            )
            result = cursor.fetchone()[0]
            conn.close()
            return result
        except psycopg2.errors.UniqueViolation:
            # Collision! The code already exists. Try again with salt.
            conn.rollback()
            print(f"  ⚠️  Collision on attempt {attempt + 1}, retrying with salt...")
    
    conn.close()
    raise Exception(f"Failed to generate unique code after {max_retries} attempts")

# Demo
test_url = "https://www.example.com/hash-demo-" + str(random.randint(0, 10**10))
code = shorten_with_hash(test_url)
print(f"✅ Shortened: {test_url}")
print(f"   Code: {code}")
print()
print("Hash approach summary:")
print("  ✅ Deterministic (same URL → same code, good for deduplication)")
print("  ⚠️  Collisions possible → need retry logic")
print("  ⚠️  Retry adds latency and DB round-trips")

✅ Shortened: https://www.example.com/hash-demo-5322834327
   Code: zy6n2hP

Hash approach summary:
  ✅ Deterministic (same URL → same code, good for deduplication)
  ⚠️  Collisions possible → need retry logic
  ⚠️  Retry adds latency and DB round-trips


---
## ✅ Approach 3: Atomic Counter + Base62 (Production Approach)

Instead of hoping hashes don't collide, we can **guarantee** uniqueness by using a
simple counter. Every new URL gets the next number, and we Base62-encode it.

**Why Redis for the counter?**
- Redis is **single-threaded** → no race conditions
- `INCR` is **atomic** → two simultaneous requests always get different values
- It's **fast** → 100k+ operations per second

```
Counter value:  1000000
Base62 encoded: 4c92
Short URL:      short.ly/4c92
```

In [8]:
r = get_redis_client()

# Start counter at 100000 so codes are always 3+ chars
COUNTER_KEY = "bitly:url_counter"
if not r.exists(COUNTER_KEY):
    r.set(COUNTER_KEY, 100000)

def shorten_with_counter(long_url: str) -> str:
    """
    Shorten a URL using an atomic Redis counter + Base62.
    
    Steps:
    1. INCR the Redis counter (atomic, returns new value)
    2. Base62-encode the counter value → short code
    3. Store the mapping in PostgreSQL
    """
    # Step 1: get next unique ID from Redis
    counter_value = r.incr(COUNTER_KEY)
    
    # Step 2: encode to Base62
    short_code = base62_encode(counter_value)
    
    # Step 3: store in database
    conn = get_db_connection()
    conn.autocommit = True
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO urls (short_code, long_url) VALUES (%s, %s) RETURNING short_code",
        (short_code, long_url)
    )
    result = cursor.fetchone()[0]
    conn.close()
    return result

# Demo: shorten 5 URLs
print("Counter-Based Shortening:")
print("=" * 60)
demo_urls = [
    "https://www.python.org/about/",
    "https://redis.io/docs/latest/",
    "https://www.postgresql.org/about/",
    "https://fastapi.tiangolo.com/",
    "https://docs.docker.com/get-started/",
]

for url in demo_urls:
    code = shorten_with_counter(url)
    counter_val = int(r.get(COUNTER_KEY))
    print(f"  counter={counter_val}  code={code}  ←  {url}")

print()
print("✅ Every code is unique — guaranteed by the atomic counter!")
print("✅ No collisions, no retries, no extra DB lookups.")

Counter-Based Shortening:
  counter=100001  code=q0V  ←  https://www.python.org/about/
  counter=100002  code=q0W  ←  https://redis.io/docs/latest/
  counter=100003  code=q0X  ←  https://www.postgresql.org/about/
  counter=100004  code=q0Y  ←  https://fastapi.tiangolo.com/


  counter=100005  code=q0Z  ←  https://docs.docker.com/get-started/

✅ Every code is unique — guaranteed by the atomic counter!
✅ No collisions, no retries, no extra DB lookups.


### Counter Batching (Scaling to Multiple Servers)

In production, multiple servers shorten URLs at the same time.
Instead of calling Redis for **every** URL, each server grabs a **batch** of counter values.

```
Server A asks Redis: "Give me 1000 IDs"  →  gets 100001–101000
Server B asks Redis: "Give me 1000 IDs"  →  gets 101001–102000
```

Each server uses its batch locally — no more Redis calls until the batch runs out.

In [9]:
class CounterBatch:
    """
    Simulates a server that grabs a batch of counter values from Redis.
    Uses IDs locally until the batch runs out, then grabs another.
    """
    def __init__(self, name: str, batch_size: int = 100):
        self.name = name
        self.batch_size = batch_size
        self.current_id = 0
        self.max_id = 0
        self.redis_calls = 0
    
    def next_id(self) -> int:
        """Get the next unique ID, fetching a new batch if needed."""
        if self.current_id >= self.max_id:
            # Batch exhausted — grab a new one from Redis
            # INCRBY atomically adds batch_size and returns the new value
            new_max = r.incrby(COUNTER_KEY, self.batch_size)
            self.current_id = new_max - self.batch_size
            self.max_id = new_max
            self.redis_calls += 1
        
        self.current_id += 1
        return self.current_id

# Simulate two servers shortening URLs
server_a = CounterBatch("Server-A", batch_size=5)
server_b = CounterBatch("Server-B", batch_size=5)

print("Counter Batching Demo (batch_size=5):")
print("=" * 50)

# Each server processes some URLs
all_ids = []
for i in range(8):
    # Alternate between servers
    server = server_a if i % 2 == 0 else server_b
    uid = server.next_id()
    code = base62_encode(uid)
    all_ids.append(uid)
    print(f"  {server.name}: id={uid}, code={code}")

print()
print(f"  Server-A Redis calls: {server_a.redis_calls}")
print(f"  Server-B Redis calls: {server_b.redis_calls}")
print(f"  All IDs unique: {len(all_ids) == len(set(all_ids))}")
print()
print("💡 With batch_size=1000, a busy server might call Redis only once per second!")

Counter Batching Demo (batch_size=5):
  Server-A: id=100006, code=q10
  Server-B: id=100011, code=q15
  Server-A: id=100007, code=q11
  Server-B: id=100012, code=q16
  Server-A: id=100008, code=q12
  Server-B: id=100013, code=q17
  Server-A: id=100009, code=q13
  Server-B: id=100014, code=q18

  Server-A Redis calls: 1
  Server-B Redis calls: 1
  All IDs unique: True

💡 With batch_size=1000, a busy server might call Redis only once per second!


---
## 📊 Comparison: Hash vs Counter

| | Hash + Base62 | Counter + Base62 |
|---|---|---|
| **Uniqueness** | Probabilistic (collisions possible) | Guaranteed |
| **Collisions** | Need retry logic | None |
| **Deduplication** | Same URL → same code (free) | Same URL → different code |
| **Predictability** | Hard to guess next code | Sequential (enumerable) |
| **Speed** | Fast (no network call) | Fast (Redis INCR is ~0.1ms) |
| **Scaling** | No coordination needed | Need shared counter (Redis) |

**In interviews**: The counter approach is preferred because it eliminates collisions entirely.
Mention the predictability concern and that you can XOR with a secret key if needed.

In [10]:
# Performance comparison: hash vs counter

# Hash approach: hash + encode + DB insert (with possible retry)
hash_times = []
for i in range(100):
    url = f"https://example.com/perf-hash-{i}-{random.randint(0,10**10)}"
    start = time.time()
    code = shorten_with_hash(url)
    hash_times.append((time.time() - start) * 1000)

# Counter approach: Redis INCR + encode + DB insert
counter_times = []
for i in range(100):
    url = f"https://example.com/perf-counter-{i}-{random.randint(0,10**10)}"
    start = time.time()
    code = shorten_with_counter(url)
    counter_times.append((time.time() - start) * 1000)

print("⏱️  Performance (100 URLs each):")
print("=" * 50)
print(f"  Hash approach:    avg {sum(hash_times)/len(hash_times):.2f} ms")
print(f"  Counter approach: avg {sum(counter_times)/len(counter_times):.2f} ms")
print()
print("💡 Both are fast. The counter wins on correctness, not speed.")

⏱️  Performance (100 URLs each):
  Hash approach:    avg 22.20 ms
  Counter approach: avg 18.89 ms

💡 Both are fast. The counter wins on correctness, not speed.


## 🧹 Cleanup

In [11]:
# Clean up the URLs we created during this notebook.
# Rule: the seed data in init.sql has exactly these 10 short_codes.
# We treat everything else in the `urls` table as test data to remove.
# IMPORTANT: delete from `clicks` first because of the foreign key constraint.
SEED_CODES = (
    'abc123', 'gH7kL9', 'Xy3mN8', 'qW5rT2', 'pL4jF6',
    'dK8vB1', 'zN9cX3', 'mR2hY7', 'tJ6wA4', 'vF1sE5',
)

conn = get_db_connection()
conn.autocommit = True
cursor = conn.cursor()

# 1) Delete clicks that reference non-seed URLs (to satisfy the FK)
cursor.execute("DELETE FROM clicks WHERE short_code NOT IN %s", (SEED_CODES,))
# 2) Delete the non-seed URL rows themselves
cursor.execute("DELETE FROM urls WHERE short_code NOT IN %s", (SEED_CODES,))
conn.close()

# Reset the Redis counter so re-running the notebook starts fresh
r = get_redis_client()
r.delete(COUNTER_KEY)
print("🧹 Cleaned up test data (seed rows preserved)")


🧹 Cleaned up test data (seed rows preserved)


### Other ID Strategies You'll See in the Wild

The counter approach is simple and correct, but it's not the only option. A few
you'll hear about in system design interviews:

- **Ticket Server** — a dedicated service (often MySQL `AUTO_INCREMENT`) hands out
  IDs. Same idea as our Redis counter, but persisted in a database.
- **Snowflake (Twitter)** — 64-bit IDs built from `[timestamp | worker_id | sequence]`.
  Every server generates unique IDs *locally* with no coordination. Great for scale.
- **UUIDs** — 128-bit random IDs, unique without coordination, but 22+ chars in
  Base62 — too long for a URL shortener.
- **Pre-generated pool (Key Generation Service)** — a batch job pre-generates
  millions of unused short codes in a table; the shorten API just claims the next
  unused one. Zero coordination on the hot path.

For a URL shortener, the Redis counter + Base62 is a perfect starting point.
You only need something fancier if a single Redis counter becomes a bottleneck
or a single point of failure you can't tolerate.


## 📚 Summary

### Key Takeaways

1. **Truncation doesn't work** — URLs that share a prefix collide immediately
2. **Hash + Base62 works** — but has a small collision probability that grows with scale
3. **Counter + Base62 is best** — atomic Redis counter guarantees uniqueness, zero collisions
4. **Base62** is the standard URL-safe encoding (62 chars: `a-z A-Z 0-9`)
5. **Counter batching** reduces Redis calls in a multi-server setup

### Next Up

In **Notebook 2**, we'll build a **redirect service** with FastAPI and add Redis caching
so redirects happen in under 1ms instead of hitting the database every time.